# LSTM Model Runner Validation

Notebook ini memvalidasi runner inference untuk checkpoint LSTM aktif tanpa menambahkan file deep learning `.py`. Runner memuat registry aktif, checkpoint PyTorch, menjalankan single dan batch prediction, lalu inverse-transform output ke km/h.

In [1]:
from __future__ import annotations

import json
import sys
import time
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import torch

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src" / "traffic_prediction").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC = PROJECT_ROOT / "src"
if not SRC.exists():
    raise RuntimeError(f"Could not locate project src directory from {Path.cwd()}")
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from traffic_prediction.data.scalers import ScalerStore
from traffic_prediction.models.lstm import LSTMModelConfig, TrafficLSTM

MODEL_ROOT = PROJECT_ROOT / "artifacts" / "models"
REGISTRY_PATH = MODEL_ROOT / "registry.json"
SEQUENCE_ROOT = PROJECT_ROOT / "artifacts" / "reports"

In [2]:
registry = json.loads(REGISTRY_PATH.read_text(encoding="utf-8"))
active_version = registry["active_model_version"]
active_entry = next(item for item in registry["models"] if item["model_version"] == active_version)
artifact_path = Path(active_entry["artifact_path"])
metadata = json.loads((artifact_path / "model_config.json").read_text(encoding="utf-8"))
checkpoint_path = Path(metadata["checkpoint_path"])
sequence_path = SEQUENCE_ROOT / active_version / "lstm_sequences.npz"
scaler_path = MODEL_ROOT / "scaler_params.joblib"

print(f"Active model: {active_version}")
print(f"Checkpoint: {checkpoint_path}")
print(f"Sequences: {sequence_path}")
print(f"Scaler: {scaler_path}")

Active model: lstm-real-20260518-032721
Checkpoint: C:\AIproject\AWAI\artifacts\models\lstm-real-20260518-032721\model.pt
Sequences: C:\AIproject\AWAI\artifacts\reports\lstm-real-20260518-032721\lstm_sequences.npz
Scaler: C:\AIproject\AWAI\artifacts\models\scaler_params.joblib


In [3]:
@dataclass(frozen=True)
class NotebookModelRunnerResult:
    scaled_prediction: np.ndarray
    kmh_prediction: np.ndarray
    latency_ms: float


class NotebookLSTMModelRunner:
    def __init__(self, artifact_path: Path, scaler_path: Path, device: str = "cpu") -> None:
        self.artifact_path = artifact_path
        self.scaler_path = scaler_path
        self.device = torch.device(device)
        self.metadata = json.loads((artifact_path / "model_config.json").read_text(encoding="utf-8"))
        self.scaler = ScalerStore.load(scaler_path)
        config_payload = dict(self.metadata["model_config"])
        config_payload["hidden_sizes"] = tuple(config_payload["hidden_sizes"])
        self.model = TrafficLSTM(LSTMModelConfig(**config_payload)).to(self.device)
        checkpoint = torch.load(self.metadata["checkpoint_path"], map_location=self.device, weights_only=False)
        self.model.load_state_dict(checkpoint["model_state_dict"])
        self.model.eval()

    def predict(self, sequence_batch: np.ndarray) -> NotebookModelRunnerResult:
        if sequence_batch.ndim != 3:
            raise ValueError("sequence_batch must have shape (batch, lookback, features)")
        started = time.perf_counter()
        tensor = torch.as_tensor(sequence_batch, dtype=torch.float32, device=self.device)
        with torch.no_grad():
            scaled = self.model(tensor).detach().cpu().numpy()
        latency_ms = (time.perf_counter() - started) * 1000
        kmh = self.inverse_transform_speed(scaled)
        return NotebookModelRunnerResult(scaled_prediction=scaled, kmh_prediction=kmh, latency_ms=latency_ms)

    def inverse_transform_speed(self, values: np.ndarray) -> np.ndarray:
        flat = values.reshape(-1, 1)
        restored = self.scaler.inverse_transform_speed(flat).reshape(values.shape)
        return np.clip(restored, 0.0, 120.0)


runner = NotebookLSTMModelRunner(artifact_path=artifact_path, scaler_path=scaler_path)
runner.metadata["model_version"]

'lstm-real-20260518-032721'

In [4]:
with np.load(sequence_path) as data:
    X_test = data["X_test"]
    y_test = data["y_test"]

single_result = runner.predict(X_test[:1])
batch_result = runner.predict(X_test[:32])
target_kmh = runner.inverse_transform_speed(y_test[:32])

single_prediction = single_result.kmh_prediction.reshape(-1).round(3).tolist()
batch_prediction = batch_result.kmh_prediction.squeeze(-1)
batch_mae = float(np.mean(np.abs(batch_prediction - target_kmh.squeeze(-1))))

print("Single prediction km/h:", single_prediction)
print("Batch prediction shape:", batch_result.kmh_prediction.shape)
print("Batch MAE on first 32 test samples:", round(batch_mae, 3))
print("Single latency ms:", round(single_result.latency_ms, 3))
print("Batch latency ms:", round(batch_result.latency_ms, 3))

Single prediction km/h: [25.415000915527344, 25.57900047302246, 25.937000274658203, 25.680999755859375]
Batch prediction shape: (32, 4, 1)
Batch MAE on first 32 test samples: 1.322
Single latency ms: 11.359
Batch latency ms: 2.467


In [5]:
report = {
    "status": "passed",
    "model_version": active_version,
    "artifact_path": str(artifact_path),
    "checkpoint_path": str(checkpoint_path),
    "framework": metadata.get("framework", "pytorch"),
    "runner_location": "notebooks/07_lstm_model_runner_validation.ipynb",
    "single_prediction_shape": list(single_result.kmh_prediction.shape),
    "batch_prediction_shape": list(batch_result.kmh_prediction.shape),
    "single_prediction_kmh": single_prediction,
    "batch_prediction_min_kmh": float(batch_result.kmh_prediction.min()),
    "batch_prediction_max_kmh": float(batch_result.kmh_prediction.max()),
    "batch_first_32_mae_kmh": batch_mae,
    "single_latency_ms": single_result.latency_ms,
    "batch_latency_ms": batch_result.latency_ms,
    "checks": {
        "checkpoint_loaded": checkpoint_path.exists(),
        "single_prediction_shape_valid": single_result.kmh_prediction.shape == (1, 4, 1),
        "batch_prediction_shape_valid": batch_result.kmh_prediction.shape == (32, 4, 1),
        "predictions_are_finite": bool(np.isfinite(batch_result.kmh_prediction).all()),
        "predictions_within_speed_bounds": bool((batch_result.kmh_prediction >= 0).all() and (batch_result.kmh_prediction <= 120).all()),
    },
}

if not all(report["checks"].values()):
    raise RuntimeError(report["checks"])

report_path = SEQUENCE_ROOT / active_version / "model_runner_validation.json"
report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
print(f"Wrote {report_path}")

Wrote C:\AIproject\AWAI\artifacts\reports\lstm-real-20260518-032721\model_runner_validation.json
